In [ ]:
# Демонстрационный ноутбук
# %%
from src.config import Config
from src.graph_manager import GraphManager
from src.nlp_extractor import ScientificNLP
from src.query_builder import QueryBuilder

# Инициализация
config = Config()
graph = GraphManager(config.NEO4J_URI, config.NEO4J_USER, config.NEO4J_PASSWORD)
nlp = ScientificNLP(config)
builder = QueryBuilder(nlp)

# %% [markdown]
# ## Загрузка тестовых данных

# %%
# Создание материалов
for material in ['Никель', 'Медь', 'Золото', 'Серебро', 'Сульфаты', 'Хлориды', 'Католит']:
    graph.create_material(material)

# Создание процессов
for process in ['Электроэкстракция', 'Кучное выщелачивание', 'Обессоливание', 'Взвешенная плавка']:
    graph.create_process(process)

# Создание эксперимента
exp_id = graph.create_experiment({
    'title': 'Тест электроэкстракции никеля',
    'description': 'Исследование скорости циркуляции католита',
    'materials': ['Никель', 'Католит'],
    'processes': ['Электроэкстракция'],
    'parameters': [
        {'name': 'Скорость потока', 'value': 0.08, 'unit': 'м/с', 'condition': 'оптимальная'},
        {'name': 'Температура', 'value': 60, 'unit': '°C', 'condition': None},
    ],
    'confidence': 0.9,
    'date': '2024-01-15'
})
print(f"Создан эксперимент: {exp_id}")

# %% [markdown]
# ## Выполнение тестовых запросов

# %%
# Запрос 1: Обессоливание
query = ("Какие методы обессоливания воды подходят, если исходная вода "
         "содержит сульфаты, хлориды по 200-300 мг/л?")

parsed = nlp.parse_query(query)
print(f"Intent: {parsed['intent']}")
print(f"Constraints: {parsed['constraints']}")

cypher = builder.build_cypher_query(query)
print(f"Cypher:\n{cypher['cypher']}")

# Выполнение
with graph.driver.session() as session:
    results = session.run(cypher['cypher'], cypher['params'])
    for record in results:
        print(dict(record))

# %% [markdown]
# ## Поиск экспертов

# %%
experts = graph.find_experts_by_topic('электроэкстракция')
for expert in experts:
    print(f"Эксперт: {expert['expert']}, Публикаций: {expert['relevant_papers']}")

# %% [markdown]
# ## Пробелы в знаниях

# %%
gaps = graph.get_knowledge_gaps()
for gap in gaps[:5]:
    print(f"Пробел: {gap['material']} + {gap['process']} -> {gap['gap_type']}")